#Setup

In [ ]:
import os, time, json, glob, random, datetime as dt

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Big Data Streaming') \
    .config('spark.sql.shuffle.partitions', 4) \
    .config('spark.ui.showConsoleProgress', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')


#Mecanismo Básico: Exemplo com Gerador Rate







## readStream

* readStream descreve a fonte de dados
* A fonte rate é um gerador embutido: N linhas por segundo, cada uma com um timestamp e value crescente
* Linhas é um Dataframe unbounded (ilimitado)

In [ ]:
linhas = spark \
    .readStream \
    .format('rate') \
    .option('rowsPerSecond', 5) \
    .load()

In [ ]:
linhas.printSchema()


root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



In [ ]:
linhas

DataFrame[timestamp: timestamp, value: bigint]

In [ ]:
linhas.isStreaming

True

## writeStream

* writeStream descreve o destino dos dados
* O sink memory cria uma tabela SQL para consulta
* start() inicia a coleta e retorna

In [ ]:
query = linhas.writeStream \
    .format('memory') \
    .queryName('p1') \
    .outputMode('append') \
    .trigger(processingTime='2 seconds') \
    .start()

In [ ]:
query.isActive


True

## Consulta dos dados

In [ ]:
spark.sql('select * from p1').show()


+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-09-17 22:42:...|    0|
|2026-09-17 22:42:...|    2|
|2026-09-17 22:42:...|    4|
|2026-09-17 22:42:...|    6|
|2026-09-17 22:42:...|    8|
|2026-09-17 22:42:...|    1|
|2026-09-17 22:42:...|    3|
|2026-09-17 22:42:...|    5|
|2026-09-17 22:42:...|    7|
|2026-09-17 22:42:...|    9|
|2026-09-17 22:42:...|   10|
|2026-09-17 22:42:...|   12|
|2026-09-17 22:42:...|   14|
|2026-09-17 22:42:...|   11|
|2026-09-17 22:42:...|   13|
|2026-09-17 22:42:...|   15|
|2026-09-17 22:42:...|   17|
|2026-09-17 22:42:...|   19|
|2026-09-17 22:42:...|   16|
|2026-09-17 22:42:...|   18|
+--------------------+-----+
only showing top 20 rows


In [ ]:
spark.sql('select count(*) as total from p1').show()

+-----+
|total|
+-----+
|  750|
+-----+



In [ ]:
# Consulta ordenada. Repita a execução algumas vezes para observar os valores mais recentes
spark.sql('select * from p1 order by timestamp desc limit 5').show()


+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-09-17 22:44:...|  749|
|2026-09-17 22:44:...|  748|
|2026-09-17 22:44:...|  747|
|2026-09-17 22:44:...|  746|
|2026-09-17 22:44:...|  745|
+--------------------+-----+



## Encerramento da coleta

* Uma consulta continua ativa até stop(), mesmo depois que a célula termina.

In [ ]:
query.stop()
query.isActive


False

# Streaming de Arquivos

## Preparação dos diretórios

* Faremos upload de novos dados em stream_input

In [ ]:
!rm -rf /content/stream_input /content/ck
!mkdir -p /content/stream_input

## Exemplo básico: armazenamento das linhas de texto recebidas

In [ ]:
linhas = spark \
    .readStream \
    .schema('linha STRING') \
    .format('text') \
    .load('/content/stream_input')

In [ ]:
query = linhas.writeStream \
    .format('memory') \
    .queryName('p2a') \
    .outputMode('append') \
    .trigger(processingTime='2 seconds') \
    .start()

## Leitura de linhas de texto
* Faça upload de arquivos para stream_input, e re-execute a consulta para cada upload
* Cada linha de cada arquivo vira uma linha da tabela ilimitada.

In [ ]:
spark.sql('select * from p2a limit 10').show(truncate=False)
spark.sql('select count(*) as linhas from p2a').show()


+--------------------------------------------------------------------------+
|linha                                                                     |
+--------------------------------------------------------------------------+
|The Project Gutenberg EBook of Noites de insomnia, offerecidas a quem não |
|póde dormir. Nº6 (de 12), by Camilo Castelo Branco                        |
|                                                                          |
|This eBook is for the use of anyone anywhere at no cost and with          |
|almost no restrictions whatsoever.  You may copy it, give it away or      |
|re-use it under the terms of the Project Gutenberg License included       |
|with this eBook or online at www.gutenberg.org                            |
|                                                                          |
|                                                                          |
|Title: Noites de insomnia, offerecidas a quem não póde dormir. Nº6 (de 12)|

In [ ]:
query.stop()
!rm -rf /content/stream_input /content/ck
!mkdir -p /content/stream_input

## Limpeza de linhas

* As linhas de dados brutos são transformadas:
  * Conversão para minúsculas
  * Expressão regular para manter apenas letras (classe \p{L} Unicode)

In [ ]:
# Observe que estamos reaproveitando o readStream do passo anterior
limpas = linhas.select(
    regexp_replace(lower(col('linha')), r'[^\p{L} ]', '').alias('linha_limpa')
)


In [ ]:
query = limpas.writeStream \
    .format('memory') \
    .queryName('p2b') \
    .outputMode('append') \
    .trigger(processingTime='2 seconds') \
    .start()

In [ ]:
spark.sql('select * from p2b limit 10').show(truncate=False)
spark.sql('select count(*) as linhas from p2b').show()


+------------------------------------------------------------------------+
|linha_limpa                                                             |
+------------------------------------------------------------------------+
|the project gutenberg ebook of noites de insomnia offerecidas a quem não|
|póde dormir nº de  by camilo castelo branco                             |
|                                                                        |
|this ebook is for the use of anyone anywhere at no cost and with        |
|almost no restrictions whatsoever  you may copy it give it away or      |
|reuse it under the terms of the project gutenberg license included      |
|with this ebook or online at wwwgutenbergorg                            |
|                                                                        |
|                                                                        |
|title noites de insomnia offerecidas a quem não póde dormir nº de       |
+------------------------

In [ ]:
query.stop()
!rm -rf /content/stream_input /content/ck
!mkdir -p /content/stream_input

## Conversão de linhas em palavras

In [ ]:
palavras = limpas.select(
    explode(
        split(col('linha_limpa'), ' ')
    ).alias('palavra')
).filter(col('palavra') != '')

In [ ]:
query = palavras.writeStream \
    .format('memory') \
    .queryName('p2c') \
    .outputMode('append') \
    .trigger(processingTime='2 seconds') \
    .start()

In [ ]:
spark.sql('select * from p2c limit 10').show()
spark.sql('select count(*) as palavras from p2c').show()

+-----------+
|    palavra|
+-----------+
|        the|
|    project|
|  gutenberg|
|      ebook|
|         of|
|     noites|
|         de|
|   insomnia|
|offerecidas|
|          a|
+-----------+

+--------+
|palavras|
+--------+
|  229884|
+--------+



In [ ]:
query.stop()
!rm -rf /content/stream_input /content/ck
!mkdir -p /content/stream_input

## Contagem de palavras

* Observe que a consulta agora é sobre a tabela completa

In [ ]:
contagem = palavras.groupBy('palavra').count()

In [ ]:
query = contagem.writeStream \
    .format('memory') \
    .queryName('p2d') \
    .outputMode('complete') \
    .trigger(processingTime='2 seconds') \
    .start()

In [ ]:
spark.sql('select * from p2d order by count desc, palavra limit 15').show()

+-------+-----+
|palavra|count|
+-------+-----+
|      a| 8364|
|    que| 7717|
|     de| 7366|
|      e| 7031|
|      o| 6454|
|    não| 3878|
|     um| 2920|
|     do| 2839|
|     da| 2432|
|     os| 2210|
|    com| 1964|
|    uma| 1824|
|    era| 1726|
|   para| 1723|
|     as| 1676|
+-------+-----+



In [ ]:
query.stop()
!rm -rf /content/stream_input /content/ck
!mkdir -p /content/stream_input

## Filtro para palavras pequenas

In [ ]:
contagem_filtrada = contagem.filter(length('palavra') > 3)

In [ ]:
query = contagem_filtrada.writeStream \
    .format('memory') \
    .queryName('p2e') \
    .outputMode('complete') \
    .trigger(processingTime='2 seconds') \
    .start()

In [ ]:
spark.sql('select * from p2e order by count desc, palavra limit 15').show()

+-----------+-----+
|    palavra|count|
+-----------+-----+
|       para|  161|
|       mais|  111|
|       como|   90|
|    project|   87|
|gutenbergtm|   56|
|       este|   55|
|       this|   47|
|       with|   47|
|      reino|   46|
|      todos|   46|
|       quem|   45|
|       seus|   45|
|       work|   45|
|       elle|   42|
|     quando|   41|
+-----------+-----+



In [ ]:
query.stop()
!rm -rf /content/stream_input /content/ck
!mkdir -p /content/stream_input

## Checkpoint

* checkpointLocation guarda duas coisas:
  * *offsets*: até onde cada fonte de dados foi lida
  * *estado*: os agregados parciais das consultas com agregação
* Com checkpoint, a consulta pode ser interrompida e retomada de onde parou

In [ ]:
query = contagem_filtrada.writeStream \
    .format('memory') \
    .queryName('p2f') \
    .outputMode('complete') \
    .option('checkpointLocation', '/content/ck/contagem') \
    .trigger(processingTime='2 seconds') \
    .start()

* Faça upload de um ou mais arquivos em stream_input, e consulte o resultado

In [ ]:
spark.sql('select * from p2f order by count desc, palavra limit 10').show()

+--------+-----+
| palavra|count|
+--------+-----+
|    para|  708|
|  rubião|  696|
|    elle|  424|
|  sophia|  339|
|    mais|  321|
|    como|  281|
|   maria|  222|
|    ella|  215|
|capitulo|  212|
|    casa|  212|
+--------+-----+



In [ ]:
query.stop()

In [ ]:
!ls /content/ck/contagem
!ls /content/ck/contagem/offsets
!tail -n 1 /content/ck/contagem/offsets/*
!tail -n 1 /content/ck/contagem/sources/0/*

commits  metadata  offsets  sources  state
0
{"logOffset":0}{"path":"file:///content/stream_input/u-55682-8","timestamp":1789686284719,"batchId":0}

### Retomada

* A mesma consulta, apontando para o **mesmo** checkpoint
* Faça upload de mais um arquivo e consulte:
  * As contagens anteriores continuam lá, restauradas do estado
  * Os arquivos antigos **não** são reprocessados

In [ ]:
query = contagem_filtrada.writeStream \
    .format('memory') \
    .queryName('p2g') \
    .outputMode('complete') \
    .option('checkpointLocation', '/content/ck/contagem') \
    .trigger(processingTime='2 seconds') \
    .start()

In [ ]:
spark.sql('select * from p2g order by count desc, palavra limit 10').show()

+-------+-----+
|palavra|count|
+-------+-----+
|   para| 1723|
|   mais|  978|
|   elle|  976|
|   como|  911|
| rubião|  696|
|   ella|  553|
| depois|  521|
|  tinha|  497|
| quando|  469|
|  muito|  466|
+-------+-----+



In [ ]:
query.stop()
!rm -rf /content/stream_input /content/ck
!mkdir -p /content/stream_input

# Tempo

Dados
* Avaliações de brinquedos da Amazon, separados por ano
* Diretório stream em http://tinyurl.com/bigdata-amz

## Tempo do evento e de processamento

* O tempo do evento vem dentro do dado original.
* O tempo de processamento é marcado na consulta com current_timestamp

In [ ]:
reviews = spark \
    .readStream \
    .schema('data_avaliacao TIMESTAMP, produto STRING, nota DOUBLE') \
    .format('csv') \
    .load('/content/stream_input')

In [ ]:
query = reviews \
    .select('data_avaliacao', 'produto', 'nota',
            current_timestamp().alias('processamento')) \
    .writeStream \
    .format('memory') \
    .queryName('t1') \
    .outputMode('append') \
    .trigger(processingTime='2 seconds') \
    .start()

* Faça upload de `reviews_2015.csv`

In [ ]:
spark.sql('select * from t1 limit 5').show(truncate=False)

+-------------------+----------+----+-----------------------+
|data_avaliacao     |produto   |nota|processamento          |
+-------------------+----------+----+-----------------------+
|2015-01-01 00:00:00|1223063151|5.0 |2026-09-17 23:12:04.057|
|2015-01-01 00:00:00|1579823645|5.0 |2026-09-17 23:12:04.057|
|2015-01-01 00:00:00|8677805966|5.0 |2026-09-17 23:12:04.057|
|2015-01-01 00:00:00|B00000IS6G|5.0 |2026-09-17 23:12:04.057|
|2015-01-01 00:00:00|B00000J048|5.0 |2026-09-17 23:12:04.057|
+-------------------+----------+----+-----------------------+



In [ ]:
spark.sql('''select min(data_avaliacao) as mais_antiga,
                     max(data_avaliacao) as mais_recente,
                     count(*) as avaliacoes,
                     count(distinct processamento) as valores_de_processamento
              from t1''').show(truncate=False)

+-------------------+-------------------+----------+------------------------+
|mais_antiga        |mais_recente       |avaliacoes|valores_de_processamento|
+-------------------+-------------------+----------+------------------------+
|2015-01-01 00:00:00|2015-12-31 00:00:00|40769     |1                       |
+-------------------+-------------------+----------+------------------------+



In [ ]:
query.stop()

!rm -rf /content/stream_input /content/ck
!mkdir -p /content/stream_input

## Janelas temporais

`window()` converte um timestamp num intervalo `{start, end}`, que serve de chave
para o `groupBy`.

In [ ]:
janelas = reviews \
    .groupBy(window('data_avaliacao', '90 days')) \
    .agg(count('*').alias('avaliacoes'),
         avg('nota').alias('nota_media'))



In [ ]:
query = janelas.writeStream \
    .format('memory') \
    .queryName('t2') \
    .outputMode('complete') \
    .trigger(processingTime='2 seconds') \
    .start()

In [ ]:
spark.sql('''select date_format(window.start, 'yyyy-MM-dd') as inicio,
                    avaliacoes,
                    round(nota_media, 2) as nota_media
             from t2 order by 1''').show()

+----------+----------+----------+
|    inicio|avaliacoes|nota_media|
+----------+----------+----------+
|2014-11-06|     12061|      4.32|
|2015-02-04|     19542|       4.3|
|2015-05-05|     16146|      4.28|
|2015-08-03|     14355|      4.28|
|2015-11-01|     12354|      4.23|
|2016-01-30|      9818|      4.29|
|2016-04-29|      8831|      4.27|
|2016-07-28|      8475|      4.23|
|2016-10-26|      8832|      4.22|
+----------+----------+----------+



In [ ]:
query.stop()

!rm -rf /content/stream_input /content/ck
!mkdir -p /content/stream_input

## Watermark



In [ ]:
janelas_wm = reviews \
    .withWatermark('data_avaliacao', '30 days') \
    .groupBy(window('data_avaliacao', '90 days')) \
    .agg(count('*').alias('avaliacoes'),
         avg('nota').alias('nota_media'))

In [ ]:
query = janelas_wm.writeStream \
    .format('memory') \
    .queryName('t3') \
    .outputMode('append') \
    .trigger(processingTime='2 seconds') \
    .start()

### Upload 1: `reviews_2015.csv`

In [ ]:
!mv /content/*.csv /content/stream_input/ 2>/dev/null
!ls /content/stream_input

reviews_2015.csv


In [ ]:
spark.sql('''select date_format(window.start, 'yyyy-MM-dd') as inicio, avaliacoes
             from t3 order by 1''').show()

+----------+----------+
|    inicio|avaliacoes|
+----------+----------+
|2014-11-06|      6024|
|2015-02-04|      9784|
|2015-05-05|      8098|
|2015-08-03|      9070|
+----------+----------+



### Upload 2: `reviews_2016.csv`

In [ ]:
!mv /content/*.csv /content/stream_input/ 2>/dev/null
!ls /content/stream_input

reviews_2015.csv  reviews_2016.csv


In [ ]:
spark.sql('''select date_format(window.start, 'yyyy-MM-dd') as inicio, avaliacoes
             from t3 order by 1''').show()

+----------+----------+
|    inicio|avaliacoes|
+----------+----------+
|2014-11-06|      6024|
|2015-02-04|      9784|
|2015-05-05|      8098|
|2015-08-03|      9070|
|2015-11-01|     12354|
|2016-01-30|      9818|
|2016-04-29|      8831|
|2016-07-28|      8475|
+----------+----------+



### Upload 3: `reviews_2015_atrasados.csv`

* Fronteira de dados avançou, novos dados para as janelas antigas não mudam mais a tabela

In [ ]:
!mv /content/*.csv /content/stream_input/ 2>/dev/null
!ls /content/stream_input

reviews_2015_atrasados.csv  reviews_2015.csv  reviews_2016.csv


In [ ]:
spark.sql('''select date_format(window.start, 'yyyy-MM-dd') as inicio, avaliacoes
             from t3 order by 1''').show()


+----------+----------+
|    inicio|avaliacoes|
+----------+----------+
|2014-11-06|      6024|
|2015-02-04|      9784|
|2015-05-05|      8098|
|2015-08-03|      9070|
|2015-11-01|     12354|
|2016-01-30|      9818|
|2016-04-29|      8831|
|2016-07-28|      8475|
+----------+----------+



In [ ]:
query.stop()

!rm -rf /content/stream_input /content/ck
!mkdir -p /content/stream_input

# Streaming de Rede



## Exemplo de consulta a API publica

In [ ]:
import requests

URL = 'https://api.exchange.coinbase.com/products/BTC-USD/ticker'

requests.get(URL, timeout=10).json()

{'ask': '76418.74',
 'bid': '76418.73',
 'volume': '5258.02755234',
 'trade_id': 1094364236,
 'price': '76418.74',
 'size': '0.00248249',
 'time': '2026-09-17T23:30:06.474838658Z',
 'rfq_volume': '93.386872'}

## Fonte de dados em Python para Spark



In [ ]:
import datetime as dt
from pyspark.sql.datasource import DataSource, SimpleDataSourceStreamReader


class LeitorTicker(SimpleDataSourceStreamReader):

    def initialOffset(self):
        return {'trade_id': 0}

    def read(self, inicio):
        t = requests.get(URL, timeout=10).json()
        instante = dt.datetime.fromisoformat(t['time'])
        return ([(instante, float(t['price']))], {'trade_id': int(t['trade_id'])})

    def readBetweenOffsets(self, inicio, fim):
        return []


class FonteBitcoin(DataSource):

    @classmethod
    def name(cls):
        return 'bitcoin'

    def schema(self):
        return 'instante TIMESTAMP, preco DOUBLE'

    def simpleStreamReader(self, schema):
        return LeitorTicker()


spark.dataSource.register(FonteBitcoin)

## Consulta de dados

O nome registrado no `name()` do DataSource é o que vai em `format()`. O resto é igual a
qualquer outra consulta de streaming.

In [ ]:
negocios = spark \
    .readStream \
    .format('bitcoin') \
    .load()

negocios.printSchema()

root
 |-- instante: timestamp (nullable = true)
 |-- preco: double (nullable = true)



In [ ]:
query = negocios.writeStream \
    .format('memory') \
    .queryName('btc') \
    .outputMode('append') \
    .trigger(processingTime='5 seconds') \
    .start()

In [ ]:
spark.sql('select * from btc order by instante desc limit 10').show(truncate=False)

+--------------------------+--------+
|instante                  |preco   |
+--------------------------+--------+
|2026-09-17 23:33:08.944932|76385.8 |
|2026-09-17 23:33:04.02252 |76381.99|
|2026-09-17 23:33:00.008874|76381.99|
|2026-09-17 23:32:54.038828|76381.99|
|2026-09-17 23:32:49.142392|76382.0 |
|2026-09-17 23:32:44.009191|76382.0 |
|2026-09-17 23:32:39.39562 |76382.0 |
|2026-09-17 23:32:37.540371|76383.02|
|2026-09-17 23:32:34.276154|76383.02|
+--------------------------+--------+



In [ ]:
query.stop()

!rm -rf /content/ck


## Agregação sobre o fluxo

- Mínimo, máximo e média por janela de 30 segundos.

- Sem watermark e em modo complete: são poucos dados e queremos ver
todas as janelas.

In [ ]:
janelas = negocios \
    .groupBy(window('instante', '30 seconds')) \
    .agg(min('preco').alias('minimo'),
         max('preco').alias('maximo'),
         avg('preco').alias('medio'),
         count('*').alias('negocios'))

query = janelas.writeStream \
    .format('memory') \
    .queryName('btc_janelas') \
    .outputMode('complete') \
    .trigger(processingTime='5 seconds') \
    .start()

In [ ]:
spark.sql('''select date_format(window.start, 'HH:mm:ss') as inicio,
                    round(minimo, 2) as minimo,
                    round(maximo, 2) as maximo,
                    round(medio, 2) as medio,
                    negocios
             from btc_janelas order by 1''').show()

+--------+--------+--------+--------+--------+
|  inicio|  minimo|  maximo|   medio|negocios|
+--------+--------+--------+--------+--------+
|23:33:30|76381.58|76387.92|76384.76|       5|
|23:34:00|76331.84|76378.63|76355.15|       6|
|23:34:30|76333.79| 76339.5|76337.12|       6|
|23:35:00|76336.68|76336.69|76336.68|       6|
|23:35:30|76336.67|76344.38|76338.72|       6|
|23:36:00|76337.18|76337.19|76337.19|       6|
|23:36:30| 76315.0|76347.38| 76338.4|       6|
|23:37:00|76320.98| 76347.2|76335.71|       6|
|23:37:30|76347.62|76373.47| 76363.1|       6|
|23:38:00|76365.64|76373.47|76368.54|       6|
|23:38:30|76365.64|76365.65|76365.64|       6|
|23:39:00|76346.94|76360.18|76353.56|       6|
|23:39:30| 76360.1|76362.72|76361.46|       6|
|23:40:00| 76360.1|76360.11| 76360.1|       6|
|23:40:30| 76360.1|76373.52|76367.13|       6|
|23:41:00|76369.91|76373.19|76371.55|       6|
|23:41:30|76351.82|76374.72|76367.86|       6|
|23:42:00|76346.94|76357.38|76351.93|       6|
|23:42:30|763

In [ ]:
query.stop()

!rm -rf /content/stream_input /content/ck
!mkdir -p /content/stream_input